In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from scipy.stats import mode
from matplotlib import pyplot as plt
import json
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report


data = pd.read_json("../../../../data/data.json")

data= data[data["capture"]== "tw2"]

data.sample(2)

# Coluna

Esse modelo usa como entrada cada coluna IQ que sabemos que existe no pacote, logo precisamos ger?-las

In [ ]:
#2-2-2-0-5-6-14-13-8-10 = 8*2 elementos em cada coluna, 2*8 de ref
# CTE = 20*8 => 160-4-8= 148 => 148*2 = 74 =>74//8 => 9 com resto 2, logo precisamos ignorar a ?ltima amostra 
data["iqs"] = data["phases"].map(lambda x: [np.array(x[i:i+8])[1:]-np.array(x[i:i+8])[:-1] for i in range(0,len(x),8)][:-1]) 

In [ ]:
# O Xgbost s? aceita classes enumeradas, precisamos de um jeito ent?o de mapear as classes na ida e na volta

target_names = [str(angle) for angle in set(data["angle"])]
foward_dict= {angle:label for label, angle in enumerate(target_names)}
backwards_dict= {label:angle for label, angle in enumerate(target_names)}

In [ ]:
with open("./dicts.json",'w') as file:
    json.dump({
        "target_names":target_names,
        "foward_dict":foward_dict,
        "backwards_dict":backwards_dict
    },file)

In [ ]:
tsamples =[]
tlabels =[]

vsamples=[]
vlabels=[]

# Separamos no dataset para que a gnt seja capaz de depois analizar o conte?do por pacote
tdataset, vdataset = train_test_split(data,test_size=0.3,random_state=10)

for row in tdataset.iterrows():
    for element in row[1]["iqs"]:
        tsamples.append(element)
        tlabels.append(row[1]["angle"])

for row in vdataset.iterrows():
    for element in row[1]["iqs"]:
        vsamples.append(element)
        vlabels.append(row[1]["angle"])

X_train = tsamples
X_test = vsamples

y_train = [foward_dict[str(label)] for label in tlabels]
y_test = [foward_dict[str(label)] for label in vlabels ]

In [ ]:
#Creating an XGBoost classifier
model = xgb.XGBClassifier(enable_categorical=True)

#Training the model on the training data
model.fit(X_train, y_train)

In [ ]:
#Making predictions on the test set
predictions = model.predict(X_test)

#Calculating accuracy
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=target_names))

In [ ]:
# Analise via pacote

results_labels= []
results_values = []
for row in vdataset.iterrows():
    pred=[]
    for element in row[1]["iqs"]:
        pred.append(model.predict([element])[0])
    results_labels.append(foward_dict[str(row[1]["angle"])])
    results_values.append(mode(pred)[0][0])

#Calculating accuracy
accuracy = accuracy_score(results_labels, results_values)

print("Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(results_labels, results_values,target_names=target_names))

In [ ]:
model.save_model("./pd_row_classifier.json")

# Elabora??o nos erros

In [ ]:
results = {str(l):[] for l in set(vlabels)}

for row in vdataset.iterrows():
    pred=[]
    for element in row[1]["iqs"]:
        pred.append(model.predict([element])[0])
    results[str(row[1]["angle"])].append(mode(pred)[0][0]*15)

In [ ]:
for key, values in results.items():
    print(f"Angulo {key}\n" )
    for judgement in set(values):
        print(f"\t {judgement} = {100*values.count(judgement)/len(values)}% ")

In [ ]:
fig, ax = plt.subplots(len(set(results.keys())),figsize=(5,30))
fig.set_tight_layout(True)

fig.supylabel("Percentage %")
fig.supxlabel("Judgement")


for i, pair in enumerate(results.items()):
    for judgement in set(pair[1]):
        ax[i].title.set_text(pair[0])
        ax[i].bar(judgement,100*pair[1].count(judgement)/len(pair[1]),0.8)
        plt.xlim(0,180)
